# UNSW-NB15 Intrusion Detection - Binary Classification (Normal vs Attack)
### Versi binary dari pipeline multi-class. Konsep pipeline tetap sama:
- Preprocessing (LabelEncoder per kolom, StandardScaler)
- KMeans Cluster-Based Undersampling untuk majority class
- BorderlineSMOTE untuk minority class
- XGBoost classifier
- SHAP feature importance + retrain Top-N
- Hyperparameter tuning

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc

from imblearn.over_sampling import BorderlineSMOTE
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import shap
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
train = pd.read_csv("UNSW_NB15_training-set.csv", header=None)
test = pd.read_csv("UNSW_NB15_testing-set.csv", header=None)

columns = [
    "id","dur","proto","service","state","spkts","dpkts","sbytes","dbytes",
    "rate","sttl","dttl","sload","dload","sloss","dloss","sinpkt","dinpkt",
    "sjit","djit","swin","stcpb","dtcpb","dwin","tcprtt","synack","ackdat",
    "smean","dmean","trans_depth","response_body_len","ct_srv_src",
    "ct_state_ttl","ct_dst_ltm","ct_src_dport_ltm","ct_dst_sport_ltm",
    "ct_dst_src_ltm","is_ftp_login","ct_ftp_cmd","ct_flw_http_mthd",
    "ct_src_ltm","ct_srv_dst","is_sm_ips_ports","attack_cat","label"
]

train.columns = columns
test.columns = columns

train = train.iloc[1:].copy()
test = test.iloc[1:].copy()

## 2. Binary Label Setup
### CHANGE: target = `label` (0 = Normal, 1 = Attack), bukan `attack_cat`

In [ ]:
# Pastikan label numerik
train['label'] = train['label'].astype(int)
test['label']  = test['label'].astype(int)

# Drop kolom non-feature
X_train = train.drop(columns=["id", "attack_cat", "label"])
y_train = train["label"]

X_test = test.drop(columns=["id", "attack_cat", "label"])
y_test = test["label"]

print("Train distribution:", Counter(y_train))
print("Test distribution :", Counter(y_test))
print("\n0 = Normal, 1 = Attack")

Train distribution: Counter({1: 119341, 0: 56000})
Test distribution : Counter({1: 45332, 0: 37000})

0 = Normal, 1 = Attack


## 3. Preprocessing
### LabelEncoder disimpan per kolom dalam dict agar tidak tertimpa

In [ ]:
categorical_cols = ["proto", "service", "state"]

label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()

    X_train[col] = le.fit_transform(X_train[col])

    # Handle unseen categories di test set
    X_test[col] = X_test[col].apply(
        lambda x: x if x in le.classes_ else le.classes_[0]
    )

    X_test[col] = le.transform(X_test[col])

    label_encoders[col] = le

print("Categorical encoding done.")
print("Encoders saved:", list(label_encoders.keys()))

Categorical encoding done.
Encoders saved: ['proto', 'service', 'state']


## 4. Scaling (Full Features)
### Untuk binary, target sudah 0/1, jadi tidak perlu LabelEncoder lagi untuk target

In [ ]:
# Scaling - fit hanya dari X_train
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Target binary - langsung pakai
y_train_bin = y_train.values
y_test_bin  = y_test.values

class_names = ['Normal', 'Attack']

print("Classes:", class_names)
print("X_train_scaled shape:", X_train_scaled.shape)
print("Class balance train:", Counter(y_train_bin))

Classes: ['Normal', 'Attack']
X_train_scaled shape: (175341, 42)
Class balance train: Counter({np.int64(1): 119341, np.int64(0): 56000})


## 5. Baseline: XGBoost tanpa Sampling (Unbalanced)

In [ ]:
print("Training baseline XGBoost (unbalanced)...")

# CHANGE: binary -> eval_metric='logloss', objective default 'binary:logistic'
model_xgb_unbalanced = XGBClassifier(
    n_estimators=300, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='binary:logistic',
    eval_metric='logloss', random_state=42, n_jobs=-1
)
model_xgb_unbalanced.fit(X_train_scaled, y_train_bin)
y_pred_unbalanced = model_xgb_unbalanced.predict(X_test_scaled)
y_proba_unbalanced = model_xgb_unbalanced.predict_proba(X_test_scaled)[:, 1]

print("\n=== Baseline (Unbalanced) ===")
print(classification_report(y_test_bin, y_pred_unbalanced,
                             target_names=class_names))
print(f"ROC-AUC : {roc_auc_score(y_test_bin, y_proba_unbalanced):.4f}")
print("Confusion matrix:")
print(confusion_matrix(y_test_bin, y_pred_unbalanced))

Training baseline XGBoost (unbalanced)...

=== Baseline (Unbalanced) ===
              precision    recall  f1-score   support

      Normal       0.97      0.74      0.84     37000
      Attack       0.82      0.98      0.89     45332

    accuracy                           0.87     82332
   macro avg       0.90      0.86      0.87     82332
weighted avg       0.89      0.87      0.87     82332

ROC-AUC : 0.9830
Confusion matrix:
[[27276  9724]
 [  856 44476]]


## 6. Hybrid Sampling: KMeans Undersampling + BorderlineSMOTE


### 6.1 Elbow untuk class Attack (majority)

In [ ]:
inertia_values = []

k_range = range(10, 101, 10)

X_attack = X_train_scaled[y_train_bin == 1]

print(f"Jumlah sample class Attack (1): {len(X_attack)}")

for k in k_range:

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(X_attack)

    inertia_values.append(km.inertia_)

plt.figure(figsize=(8,5))

plt.plot(k_range, inertia_values, marker='o')

plt.xlabel("Number of Clusters (k)")
plt.ylabel("WCSS / Inertia")
plt.title("Elbow Method - Class Attack")

plt.show()

### 6.3 KMeans Undersample function (sama persis dengan versi multi-class)

In [ ]:
def kmeans_undersample(
    X,
    y,
    sampling_strategy,
    cluster_strategy,
    random_state=42
):

    X_result, y_result = [], []

    unique_classes = np.unique(y)

    rng = np.random.RandomState(random_state)

    for cls in unique_classes:

        # =====================================================
        # AMBIL DATA PER CLASS
        # =====================================================
        mask = (y == cls) if not hasattr(y, 'values') else (y.values == cls)

        X_cls = X[mask]

        # target jumlah setelah undersampling
        target_n = sampling_strategy.get(cls, len(X_cls))

        # jumlah cluster khusus tiap class
        n_cluster_cls = cluster_strategy.get(cls, 50)

        # =====================================================
        # MINORITY CLASS -> AMBIL SEMUA
        # =====================================================
        if len(X_cls) <= target_n:

            X_result.append(X_cls)
            y_result.extend([cls] * len(X_cls))

        # =====================================================
        # MAJORITY CLASS -> KMEANS CLUSTER SAMPLING
        # =====================================================
        else:

            print(f"\nClass : {cls}")
            print(f"Original samples : {len(X_cls)}")
            print(f"Target samples   : {target_n}")
            print(f"Clusters used    : {n_cluster_cls}")

            # KMeans clustering
            km = KMeans(
                n_clusters=n_cluster_cls,
                random_state=random_state,
                n_init=10,
                max_iter=300
            )

            cluster_labels = km.fit_predict(X_cls)

            # jumlah sampel yang diambil per cluster
            samples_per_cluster = target_n // n_cluster_cls

            selected_indices = []

            # =====================================================
            # SAMPLING TIAP CLUSTER
            # =====================================================
            for cluster_id in range(n_cluster_cls):

                cluster_idx = np.where(cluster_labels == cluster_id)[0]

                # jika isi cluster <= target per cluster
                if len(cluster_idx) <= samples_per_cluster:

                    selected_indices.extend(cluster_idx)

                else:

                    chosen = rng.choice(
                        cluster_idx,
                        size=samples_per_cluster,
                        replace=False
                    )

                    selected_indices.extend(chosen)

            selected_indices = np.array(selected_indices)

            # =====================================================
            # JIKA JUMLAH MASIH KURANG
            # =====================================================
            if len(selected_indices) < target_n:

                remaining = np.setdiff1d(
                    np.arange(len(X_cls)),
                    selected_indices
                )

                extra = rng.choice(
                    remaining,
                    size=target_n - len(selected_indices),
                    replace=False
                )

                selected_indices = np.concatenate([
                    selected_indices,
                    extra
                ])

            X_result.append(X_cls[selected_indices])

            y_result.extend([cls] * len(selected_indices))

    # =====================================================
    # GABUNGKAN SEMUA CLASS
    # =====================================================
    X_out = np.vstack(X_result)
    y_out = np.array(y_result)

    return X_out, y_out


In [ ]:
from collections import Counter
from sklearn.metrics import classification_report
from xgboost import XGBClassifier
from imblearn.over_sampling import BorderlineSMOTE

# =========================================================
# PARAMETER
# =========================================================

TARGET_PER_CLASS = 65000

sampling_strategy_under = {
    1: TARGET_PER_CLASS,   # Attack
}

cluster_strategy = {
    1: 40,
}

sampling_strategy_smote = {
    0: TARGET_PER_CLASS,   # Normal
}

experiments = {
    "Hybrid Sampling": "hybrid",
    "KMeans Only": "kmeans",
    "BorderlineSMOTE Only": "smote"
}

# =========================================================
# LOOP EKSPERIMEN
# =========================================================

for exp_name, exp_type in experiments.items():

    print("=" * 70)
    print(exp_name)
    print("=" * 70)

    # =====================================================
    # SAMPLING
    # =====================================================

    if exp_type == "hybrid":

        # Step 1 : KMeans Undersampling
        X_sample, y_sample = kmeans_undersample(
            X_train_scaled,
            y_train_bin,
            sampling_strategy=sampling_strategy_under,
            cluster_strategy=cluster_strategy,
            random_state=42
        )

        # Step 2 : BorderlineSMOTE
        smote = BorderlineSMOTE(
            kind='borderline-2',
            sampling_strategy=sampling_strategy_smote,
            k_neighbors=3,
            random_state=42
        )

        X_sample, y_sample = smote.fit_resample(
            X_sample,
            y_sample
        )

    elif exp_type == "kmeans":

        X_sample, y_sample = kmeans_undersample(
            X_train_scaled,
            y_train_bin,
            sampling_strategy=sampling_strategy_under,
            cluster_strategy=cluster_strategy,
            random_state=42
        )

    elif exp_type == "smote":

        smote = BorderlineSMOTE(
            kind='borderline-2',
            sampling_strategy=sampling_strategy_smote,
            k_neighbors=3,
            random_state=42
        )

        X_sample, y_sample = smote.fit_resample(
            X_train_scaled,
            y_train_bin
        )

    # =====================================================
    # DISTRIBUSI KELAS
    # =====================================================

    print("\nClass Distribution:")
    print(Counter(y_sample))

    # =====================================================
    # TRAIN XGBOOST
    # =====================================================

    model = XGBClassifier(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='logloss',   # binary classification
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_sample, y_sample)

    # =====================================================
    # PREDIKSI
    # =====================================================

    y_pred = model.predict(X_test_scaled)

    # =====================================================
    # HASIL
    # =====================================================

    print("\nClassification Report:")
    print(classification_report(
        y_test_bin,
        y_pred,
        target_names=["Normal", "Attack"]
    ))

Hybrid Sampling

Class : 1
Original samples : 119341
Target samples   : 65000
Clusters used    : 40

Class Distribution:
Counter({np.int64(0): 65000, np.int64(1): 65000})

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.83      0.89     37000
      Attack       0.87      0.97      0.92     45332

    accuracy                           0.90     82332
   macro avg       0.91      0.90      0.90     82332
weighted avg       0.91      0.90      0.90     82332

KMeans Only

Class : 1
Original samples : 119341
Target samples   : 65000
Clusters used    : 40

Class Distribution:
Counter({np.int64(1): 65000, np.int64(0): 56000})

Classification Report:
              precision    recall  f1-score   support

      Normal       0.95      0.81      0.88     37000
      Attack       0.86      0.97      0.91     45332

    accuracy                           0.90     82332
   macro avg       0.91      0.89      0.89     82332
weighted avg   